<a href="https://colab.research.google.com/github/avindumihisara0229-code/ErgoSense/blob/Ramzan/Drowsiness_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install necessary libraries
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.14 opencv-python-headless numpy tqdm

Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
  Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.7 kB)
Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (35.7 MB)


In [2]:
# Import all required libraries
import os
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm

print(f"MediaPipe version: {mp.__version__}")
print("All libraries imported successfully")

MediaPipe version: 0.10.14
All libraries imported successfully


## Connect Google Drive

In [3]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Define dataset paths
DATASET_PATH = '/content/drive/MyDrive/dataset_new'
TRAIN_PATH = os.path.join(DATASET_PATH, 'train')
TEST_PATH = os.path.join(DATASET_PATH, 'test')

# Verify dataset exists
if os.path.exists(DATASET_PATH):
    print("Dataset found!")
    print("\nTrain folders:", os.listdir(TRAIN_PATH))
    print("Test folders:", os.listdir(TEST_PATH))
else:
    print("ERROR: Dataset not found. Please upload dataset to Google Drive.")

Dataset found!

Train folders: ['.DS_Store', 'yawn', 'no_yawn', 'Closed', 'Open']
Test folders: ['Open', 'yawn', 'Closed', 'no_yawn']


## Initialize MediaPipe Face Mesh

In [5]:
# Initialize MediaPipe Face Mesh using direct import
from mediapipe.python.solutions import face_mesh

face_mesh_detector = face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

# Define landmark indices for eyes and mouth
# Left eye landmarks
LEFT_EYE = [362, 385, 387, 263, 373, 380]
# Right eye landmarks
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
# Mouth landmarks
MOUTH = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 88]

print("MediaPipe Face Mesh initialized successfully")

MediaPipe Face Mesh initialized successfully


## Define EAR and MAR calculation function

In [6]:
def calculate_distance(point1, point2):
    """Calculate Euclidean distance between two points"""
    return np.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

def calculate_ear(eye_landmarks):
    """Calculate Eye Aspect Ratio (EAR)"""
    # Vertical distances
    vertical1 = calculate_distance(eye_landmarks[1], eye_landmarks[5])
    vertical2 = calculate_distance(eye_landmarks[2], eye_landmarks[4])

    # Horizontal distance
    horizontal = calculate_distance(eye_landmarks[0], eye_landmarks[3])

    # EAR formula
    if horizontal == 0:
        return 0
    ear = (vertical1 + vertical2) / (2.0 * horizontal)
    return ear

def calculate_mar(mouth_landmarks):
    """Calculate Mouth Aspect Ratio (MAR)"""
    # Vertical distances
    vertical1 = calculate_distance(mouth_landmarks[2], mouth_landmarks[10])
    vertical2 = calculate_distance(mouth_landmarks[4], mouth_landmarks[8])
    vertical3 = calculate_distance(mouth_landmarks[6], mouth_landmarks[14])

    # Horizontal distance
    horizontal = calculate_distance(mouth_landmarks[0], mouth_landmarks[12])

    # MAR formula
    if horizontal == 0:
        return 0
    mar = (vertical1 + vertical2 + vertical3) / (3.0 * horizontal)
    return mar

print("Feature calculation functions defined successfully")

Feature calculation functions defined successfully


## Define feature extraction function

In [7]:
def extract_features_from_image(image_path):
    """Extract EAR and MAR features from a single image"""
    try:
        # Read image
        image = cv2.imread(image_path)
        if image is None:
            return None

        # Convert to RGB
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Process with MediaPipe
        results = face_mesh.process(rgb_image)

        if not results.multi_face_landmarks:
            return None

        # Get landmarks
        landmarks = results.multi_face_landmarks[0].landmark
        h, w = image.shape[:2]

        # Extract eye landmarks
        left_eye_points = [(landmarks[i].x * w, landmarks[i].y * h) for i in LEFT_EYE]
        right_eye_points = [(landmarks[i].x * w, landmarks[i].y * h) for i in RIGHT_EYE]

        # Extract mouth landmarks
        mouth_points = [(landmarks[i].x * w, landmarks[i].y * h) for i in MOUTH]

        # Calculate ratios
        left_ear = calculate_ear(left_eye_points)
        right_ear = calculate_ear(right_eye_points)
        avg_ear = (left_ear + right_ear) / 2.0

        mar = calculate_mar(mouth_points)

        return [avg_ear, mar]

    except Exception as e:
        return None


print("Feature extraction function defiend successsfully")

Feature extraction function defiend successsfully


## Sample Data Loading and Testing

In [8]:
# Sample dataset function (for today's demo)
def load_dataset_sample(base_path, max_per_folder=10):
    """Load only a few images per folder for quick testing"""
    features = []
    labels = []

    label_map = {
        'Open': 0,
        'no_yawn': 0,
        'Closed': 1,
        'yawn': 1
    }

    folders = os.listdir(base_path)

    for folder in folders:
        if folder not in label_map:
            continue

        folder_path = os.path.join(base_path, folder)
        if not os.path.isdir(folder_path):
            continue

        print(f"Processing {folder} folder (max {max_per_folder} images)...")
        images = os.listdir(folder_path)[:max_per_folder]

        for img_name in tqdm(images):
            img_path = os.path.join(folder_path, img_name)
            feature_vector = extract_features_from_image(img_path)

            if feature_vector is not None:
                features.append(feature_vector)
                labels.append(label_map[folder])

    return np.array(features), np.array(labels)

# Use sample function for demo
print("\nExtracting features from TRAINING data (SAMPLE for demo)...")
X_train, y_train = load_dataset_sample(TRAIN_PATH, max_per_folder=10)

print("\nExtracting features from TEST data (SAMPLE for demo)...")
X_test, y_test = load_dataset_sample(TEST_PATH, max_per_folder=10)

print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Alert samples (train): {np.sum(y_train == 0)}")
print(f"Drowsy samples (train): {np.sum(y_train == 1)}")


Extracting features from TRAINING data (SAMPLE for demo)...
Processing yawn folder (max 10 images)...


100%|██████████| 10/10 [00:02<00:00,  4.06it/s]


Processing no_yawn folder (max 10 images)...


100%|██████████| 10/10 [00:02<00:00,  3.43it/s]


Processing Closed folder (max 10 images)...


...

Processing Closed folder (max 10 images)...


100%|██████████| 10/10 [00:00<00:00, 33.76it/s]


Processing no_yawn folder (max 10 images)...


100%|██████████| 10/10 [00:00<00:00, 31.33it/s]


Training samples: 0
Test samples: 0
Alert samples (train): 0
Drowsy samples (train): 0
